# Install Zero-DCE project

In [ ]:
%cd /content/
!rm -rf CISC473Project
!git clone https://github.com/woong5115/CISC473Project.git

zero_dce_dir = '/content/CISC473Project/models/Zero-DCE/'


/content
Cloning into 'cisc473-g1'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 110 (delta 32), reused 105 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 11.90 MiB | 31.72 MiB/s, done.
Resolving deltas: 100% (32/32), done.


# Install project dependencies

In [2]:
# 1. Install Miniconda
!wget -qO miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_4.12.0-Linux-x86_64.sh
!bash miniconda.sh -b -f -p /usr/local

# 2. Install PyTorch 1.0.0, Torchvision 0.2.1, and OpenCV directly via pip inside our Python 3.7 environment
!pip uninstall --yes torch torchvision
!pip install torch==1.0.0 torchvision==0.2.1 -f https://download.pytorch.org/whl/cpu/stable.html
!pip install opencv-python
!pip uninstall --yes pillow
!pip install pillow==6.2.2
!pip install -q gdown # For downloading data

PREFIX=/usr/local
Unpacking payload ...
Solving environment: | / - done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - _libgcc_mutex==0.1=main
    - _openmp_mutex==4.5=1_gnu
    - brotlipy==0.7.0=py37h27cfd23_1003
    - ca-certificates==2022.3.29=h06a4308_1
    - certifi==2021.10.8=py37h06a4308_2
    - cffi==1.15.0=py37hd667e15_1
    - charset-normalizer==2.0.4=pyhd3eb1b0_0
    - colorama==0.4.4=pyhd3eb1b0_0
    - conda-content-trust==0.1.1=pyhd3eb1b0_0
    - conda-package-handling==1.8.1=py37h7f8727e_0
    - conda==4.12.0=py37h06a4308_0
    - cryptography==36.0.0=py37h9ce1e76_0
    - idna==3.3=pyhd3eb1b0_0
    - ld_impl_linux-64==2.35.1=h7274673_9
    - libffi==3.3=he6710b0_2
    - libgcc-ng==9.3.0=h5101ec6_17
    - libgomp==9.3.0=h5101ec6_17
    - libstdcxx-ng==9.3.0=hd4cf53a_17
    - ncurses==6.3=h7f8727e_2
    - openssl==1.1.1n=h7f8727e_0
    - pip==21.2.2=py37h06a4308_0
    - pycosat==0.6.3=py37h27cfd23_0
    - pycparser==2.21=pyhd3eb

     |████████████████████████████████| 73.8 MB 53 kB/s 
Found existing installation: Pillow 9.5.0
Uninstalling Pillow-9.5.0:
  Successfully uninstalled Pillow-9.5.0
     |████████████████████████████████| 2.1 MB 8.8 MB/s 


     |████████████████████████████████| 109 kB 12.0 MB/s 


# Ingest

1. Load data
2. Filter out unwanted images from the test dataset like in the paper
3. Resize images like in the paper

In [3]:
import os
import shutil
import gdown

# Google Drive file IDs from the URLs provided
part1_id = '1HiLtYiyT9R7dR9DRTLRlUUrAicC4zzWN'
part2_id = '16VoHNPAZ5Js19zspjFOsKiGRrfkDgHoN'

# Download Part 1 (on standby for now)
print("Downloading Dataset Part 1... ")
part1_output = '/content/Dataset_Part1.zip'
gdown.download(id=part1_id, output=part1_output, quiet=True)

# Download Part 2
print("Downloading Dataset Part 2... ")
part2_output = '/content/Dataset_Part2.rar'
gdown.download(id=part2_id, output=part2_output, quiet=True)

'/content/Dataset_Part2.rar'

In [4]:
# Extract Part 1
train_extract_dir = '/content/sice_part1_dataset_extracted'
os.makedirs(train_extract_dir, exist_ok=True)
print("Extracting Dataset Part 1... ")
!unrar x -inul "{part1_output}" "{train_extract_dir}"

# Extract Part 2
test_extract_dir = '/content/sice_part2_dataset_extracted'
os.makedirs(test_extract_dir, exist_ok=True)
print("Extracting Dataset Part 2... ")
!unrar x -inul "{part2_output}" "{test_extract_dir}"
print("Done!")

Extracting Dataset Part 1... 
Extracting Dataset Part 2... 
Done!


In [5]:
import os
import glob
from collections import Counter

test_search_dir = "/content/sice_part2_dataset_extracted/Dataset_Part2"
test_sequence_folders = sorted([f for f in glob.glob(os.path.join(test_search_dir, '*')) if os.path.isdir(f)])

# Count files in each directory
lengths = []
labels_dir = None
other_dirs = []

for seq in test_sequence_folders:
    folder_name = os.path.basename(seq)
    images = sorted(glob.glob(os.path.join(seq, '*.*')))
    num_images = len(images)

    if folder_name.lower() == 'label':
        labels_dir = (seq, num_images)
        continue

    lengths.append(num_images)

    # Print sequences that do not meet the 7 or 9 length rules
    if num_images not in [7, 9]:
        other_dirs.append((folder_name, num_images))

# Summarize the sequence lengths
length_counts = Counter(lengths)
print(f"Total sequence folders (excluding Label if any): {len(lengths)}")
print("Distribution of sequence lengths (number of images per folder):")
for length, count in sorted(length_counts.items()):
    print(f"  Length {length}: {count} sequences")

Total sequence folders (excluding Label if any): 229
Distribution of sequence lengths (number of images per folder):
  Length 5: 3 sequences
  Length 6: 14 sequences
  Length 7: 113 sequences
  Length 8: 2 sequences
  Length 9: 97 sequences


Each sequence is a dataset subfolder containing the same image at different brightnesses. In the Zero-DCE paper, they mention that they "choose the first three (resp. four) low-light images if there are seven (resp. nine) images" but don't specify how they handle sequences of other lengths. To reach the final dataset size of 767, I can only assume that the first two are taken from lengths 5 and 6, and the first three are taken from length 8.

In [6]:
import glob
import os
from PIL import Image

# Define target directories
test_data_dir = zero_dce_dir + "Zero-DCE_code/data/test_data/sice_part2"
train_data_dir = zero_dce_dir + "Zero-DCE_code/data/train_data"
os.makedirs(test_data_dir, exist_ok=True)
os.makedirs(train_data_dir, exist_ok=True)

# --- PROCESS PART 2 (TEST DATA) WITH FILTERING ---
test_search_dir = os.path.join(test_extract_dir, 'Dataset_Part2')
if not os.path.exists(test_search_dir):
    test_search_dir = test_extract_dir

test_sequence_folders = sorted([f for f in glob.glob(os.path.join(test_search_dir, '*')) if os.path.isdir(f)])

filtered_test_images = []
for seq in test_sequence_folders:
    folder_name = os.path.basename(seq)
    if folder_name.lower() == 'label':
        continue
    images = sorted(glob.glob(os.path.join(seq, '*.*')))
    if not images:
        continue
    length = len(images)
    if length in [5, 6]:
        selected_images = images[:2]
    elif length == 7:
        selected_images = images[:3]
    elif length == 8:
        selected_images = images[:3]
    elif length == 9:
        selected_images = images[:4]
    else:
        selected_images = []
    filtered_test_images.extend(selected_images)

print(f"Total filtered test images found: {len(filtered_test_images)}")
assert len(filtered_test_images) == 767, f"Error: Expected 767 images, but got {len(filtered_test_images)}."

# Resize and copy test images
for idx, img_path in enumerate(filtered_test_images):
    try:
        with Image.open(img_path) as img:
            resized_img = img.resize((1200, 900), Image.ANTIALIAS if hasattr(Image, 'ANTIALIAS') else Image.Resampling.LANCZOS)
            seq_name = os.path.basename(os.path.dirname(img_path))
            base_name = os.path.basename(img_path)
            target_path = os.path.join(test_data_dir, f"{seq_name}_{base_name}")
            resized_img.save(target_path)
    except Exception as e:
        print(f"Error processing test image {img_path}: {e}")

print(f"Successfully loaded {len(os.listdir(test_data_dir))} images to {test_data_dir}")

# --- PROCESS PART 1 (TRAIN DATA) WITHOUT FILTERING ---
train_search_dir = os.path.join(train_extract_dir, 'Dataset_Part1')
if not os.path.exists(train_search_dir):
    train_search_dir = train_extract_dir

train_sequence_folders = sorted([f for f in glob.glob(os.path.join(train_search_dir, '*')) if os.path.isdir(f)])

train_images = []
for seq in train_sequence_folders:
    folder_name = os.path.basename(seq)
    if folder_name.lower() == 'label':
        continue
    images = sorted(glob.glob(os.path.join(seq, '*.*')))
    train_images.extend(images)

print(f"Total train images found (no filtering): {len(train_images)}")

# Resize and copy train images
for idx, img_path in enumerate(train_images):
    try:
        with Image.open(img_path) as img:
            resized_img = img.resize((1200, 900), Image.ANTIALIAS if hasattr(Image, 'ANTIALIAS') else Image.Resampling.LANCZOS)
            seq_name = os.path.basename(os.path.dirname(img_path))
            base_name = os.path.basename(img_path)
            target_path = os.path.join(train_data_dir, f"{seq_name}_{base_name}")
            resized_img.save(target_path)
    except Exception as e:
        print(f"Error processing train image {img_path}: {e}")

print(f"Successfully loaded {len(os.listdir(train_data_dir))} images to {train_data_dir}")

Total filtered test images found: 767
Successfully loaded 767 images to /content/cisc473-g1/models/Zero-DCE/Zero-DCE_code/data/test_data/sice_part2
Total train images found (no filtering): 3021
Successfully loaded 3021 images to /content/cisc473-g1/models/Zero-DCE/Zero-DCE_code/data/train_data


# Evaluate

Metrics: PSNR, SSIM

In [ ]:
import cv2
import os
import numpy as np
import glob
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

original_images_dir = zero_dce_dir + "Zero-DCE_code/data/test_data/sice_part2"
enhanced_images_dir = zero_dce_dir + "Zero-DCE_code/data/result/sice_part2"

original_images = sorted(glob.glob(os.path.join(original_images_dir, "*.*")))
enhanced_images = sorted(glob.glob(os.path.join(enhanced_images_dir, "*.*")))

seeds = [42, 123, 999]

psnr_scores = []
ssim_scores = []

%cd /content/CISC473Project/models/Zero-DCE/Zero-DCE_code/

for seed in seeds:
  # !python lowlight_train.py --seed {seed}
  # !rm -rf ./data/result/
  # !python lowlight_test.py

  for orig_path, enh_path in zip(original_images, enhanced_images):
      img_orig = cv2.imread(orig_path)
      img_enh = cv2.imread(enh_path)

      if img_orig is None or img_enh is None:
          continue

      # Calculate metrics
      p_val = psnr(img_orig, img_enh)
      # Compute multi-channel SSIM
      s_val = ssim(img_orig, img_enh, channel_axis=-1)

      psnr_scores.append(p_val)
      ssim_scores.append(s_val)

mean_psnr = np.mean(psnr_scores)
std_psnr = np.std(psnr_scores)
mean_ssim = np.mean(ssim_scores)
std_ssim = np.std(ssim_scores)

print(f"Evaluation results for sice_part2 subset:")
print(f"Mean PSNR: {mean_psnr:.4f}")
print(f"Std PSNR: {std_psnr:.4f}")
print(f"Mean SSIM: {mean_ssim:.4f}")
print(f"Std SSIM: {std_ssim:.4f}")

/content/cisc473-g1/models/Zero-DCE/Zero-DCE_code
Evaluation results for sice_part2 subset:
Mean PSNR: 12.2242
Std PSNR: 1.5094
Mean SSIM: 0.5328
Std SSIM: 0.1555
